# Oferta habitacional de Lucas do Rio Verde (MT): entregas, investimento e demanda reprimida

**Autora:** Thamires Martins da Silva
**Fonte dos dados:** Dataset curado a partir de fontes públicas (Prefeitura de Lucas do Rio Verde, MT Par/SiHab-MT, Ministério das Cidades, Caixa Econômica Federal)
**Ferramentas:** Python (pandas, matplotlib)

---

## Contexto

Este é o segundo projeto de uma análise em duas camadas sobre habitação em Lucas do Rio Verde:

- **Camada 1** — projetou a *demanda* futura por moradia a partir do crescimento populacional (1.100 a 1.800 novas unidades/ano até 2035).
- **Camada 2 (este notebook)** — mede a *oferta* real: quantas moradias de interesse social a cidade efetivamente entregou, a quem, e como isso se compara à demanda.

> **Pergunta de negócio:** A oferta de habitação de interesse social em Lucas do Rio Verde acompanha a demanda da população que precisa dela?

## Sobre o dataset

Diferente da Camada 1 (que usou API do IBGE), este dataset foi **construído manualmente** a partir de fontes públicas — não existe base pronta de habitação municipal. Cada registro tem a URL da fonte, o que torna a análise auditável. Construir dado onde ele não vem pronto é parte central do trabalho de análise.

**Escopo:** apenas habitação de interesse social (subsídio público). Empreendimentos de alto padrão ficam de fora. O único caso de renda mais alta (Parque Primavera) recebe uma flag para poder ser separado.

## 1. Configuração do ambiente

Carga das bibliotecas e definição da paleta de cores (a mesma da Camada 1, para consistência visual entre os dois projetos).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paleta consistente com a Camada 1
cores = {'Conservador': '#7fcdbb', 'Base': '#2c7fb8', 'Otimista': '#253494'}
COR_ENTREGUE = '#2c7fb8'
COR_OBRA = '#7fcdbb'
COR_DEMANDA = '#d95f0e'

base = Path.cwd()
print('Pasta de trabalho:', base)

## 2. Carga e verificação dos dados

Dois datasets: `oferta_habitacional_lrv.csv` (empreendimentos) e `demanda_habitacional_lrv.csv` (fila de espera).

O dataset de oferta usa **colunas separadas por status** (`unidades_entregues`, `unidades_em_obra`, `unidades_previstas`), o que impede que unidades apenas planejadas sejam contadas como entregues — distinção crítica para não inflar os números.

In [ ]:
oferta = pd.read_csv(base / 'oferta_habitacional_lrv.csv')
demanda = pd.read_csv(base / 'demanda_habitacional_lrv.csv')

# Normaliza o indicador de escopo para tornar a análise reproduzível
# mesmo quando o CSV é carregado com valores booleanos ou texto.
oferta['interesse_social_estrito'] = (
    oferta['interesse_social_estrito'].astype(str).str.lower().eq('true')
)
oferta_social = oferta[oferta['interesse_social_estrito']].copy()

print(f'Oferta: {len(oferta)} empreendimentos no dataset')
print(f'No escopo estrito de habitação social: {len(oferta_social)} empreendimentos')
print(f'Demanda: {len(demanda)} registro(s) da fila histórica')
print()
print('--- Verificação de qualidade (oferta) ---')
for c in ['unidades_entregues', 'unidades_em_obra', 'status', 'ano_entrega']:
    print(f'  {c}: {oferta[c].isna().sum()} nulos')
print()
print('Status possíveis:', oferta['status'].unique().tolist())
print('Entregas registradas, categorias mistas:', int(oferta_social['unidades_entregues'].sum()))
print('  Casas/apartamentos entregues:', int(oferta_social.loc[oferta_social['tipo'].isin(['casa', 'apartamento']), 'unidades_entregues'].sum()))
print('  Lotes urbanizados entregues:', int(oferta_social.loc[oferta_social['tipo'].eq('lote urbanizado'), 'unidades_entregues'].sum()))
print('Moradias em obra no escopo social estrito:', int(oferta_social['unidades_em_obra'].sum()))
print('Unidades em obra fora do escopo social estrito:', int(oferta.loc[~oferta['interesse_social_estrito'], 'unidades_em_obra'].sum()))

## 3. Entregas ao longo do tempo

A pergunta é quantas **casas e apartamentos** de interesse social foram entregues por ano. Lotes urbanizados são mostrados separadamente, porque não equivalem a uma moradia pronta.

Aqui usamos apenas registros com `status == 'entregue'`. Unidades em obra ou previstas não entram na contagem de entregas realizadas.

In [ ]:
entregues = oferta_social[oferta_social['status'] == 'entregue'].copy()
entregues['ano_entrega'] = entregues['ano_entrega'].astype(int)

# Separa moradias prontas de lotes urbanizados.
entregues_moradias = entregues[entregues['tipo'].isin(['casa', 'apartamento'])].copy()
entregues_lotes = entregues[entregues['tipo'] == 'lote urbanizado'].copy()

por_ano = entregues_moradias.groupby('ano_entrega')['unidades_entregues'].sum()
por_ano_lotes = entregues_lotes.groupby('ano_entrega')['unidades_entregues'].sum()

# Reindexa para mostrar todos os anos, inclusive aqueles sem entrega.
anos_completos = range(
    int(entregues['ano_entrega'].min()),
    int(entregues['ano_entrega'].max()) + 1
)
por_ano = por_ano.reindex(anos_completos, fill_value=0)
por_ano_lotes = por_ano_lotes.reindex(anos_completos, fill_value=0)

total_moradias_entregues = int(por_ano.sum())
total_lotes_entregues = int(por_ano_lotes.sum())

print('Casas e apartamentos entregues por ano:')
print(por_ano.to_string())
print('\nLotes urbanizados entregues por ano:')
print(por_ano_lotes.to_string())
print(f'\nTotal de casas/apartamentos: {total_moradias_entregues}')
print(f'Total de lotes urbanizados: {total_lotes_entregues}')
print(f'Período analisado: {por_ano.index.min()}–{por_ano.index.max()}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(por_ano.index))
largura = 0.38
b1 = ax.bar(x - largura / 2, por_ano.values, largura,
            color=COR_ENTREGUE, label='Casas/apartamentos', zorder=3)
b2 = ax.bar(x + largura / 2, por_ano_lotes.values, largura,
            color=COR_OBRA, label='Lotes urbanizados', zorder=3)

for barras in [b1, b2]:
    for barra in barras:
        altura = barra.get_height()
        if altura > 0:
            ax.annotate(f'{int(altura)}', (barra.get_x() + barra.get_width() / 2, altura),
                        textcoords='offset points', xytext=(0, 5),
                        ha='center', fontsize=9, fontweight='bold')

ax.set_title('Lucas do Rio Verde: entregas de habitação social por ano\n'
             'Casas/apartamentos e lotes urbanizados em categorias separadas', fontsize=12)
ax.set_xlabel('Ano de entrega')
ax.set_ylabel('Unidades ou lotes entregues')
ax.set_xticks(x)
ax.set_xticklabels(por_ano.index)
ax.legend()
ax.grid(True, axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

### Leitura: entregas irregulares e categorias diferentes

As entregas são concentradas em anos de conclusão de empreendimentos, e não formam um fluxo constante. O gráfico distingue **casas/apartamentos prontos** de **lotes urbanizados**: um lote entregue amplia a oferta de terreno urbanizado, mas não significa que uma família já recebeu uma casa pronta.

Também é importante não interpretar anos sem entrega registrada como prova de que nenhuma família foi atendida por outros programas ou canais.

## 4. O que ainda vem: pipeline em obras

As entregas passadas contam metade da história. A outra metade é o que está sendo construído agora — o pipeline que definirá a oferta dos próximos anos.

In [ ]:
em_obra_total = oferta[oferta['status'] == 'em obra'].copy()
em_obra = em_obra_total[em_obra_total['interesse_social_estrito']].copy()
em_obra_fora_escopo = em_obra_total[~em_obra_total['interesse_social_estrito']].copy()

total_entregue = int(entregues['unidades_entregues'].sum())  # inclui lotes urbanizados
total_obra = int(em_obra['unidades_em_obra'].sum())          # somente escopo social estrito
total_obra_fora_escopo = int(em_obra_fora_escopo['unidades_em_obra'].sum())

print('--- Entregue, com categorias separadas ---')
print(f'Casas/apartamentos entregues: {total_moradias_entregues}')
print(f'Lotes urbanizados entregues:  {total_lotes_entregues}')
print(f'Total registrado em entregas de categorias mistas: {total_entregue}')
print()
print(f'Moradias em obra no escopo social estrito: {total_obra}')
print(f'Unidades em obra fora do escopo social estrito: {total_obra_fora_escopo}')
print()
print('--- Empreendimentos de habitação social em obra ---')
for _, r in em_obra.iterrows():
    print(f'  {r["empreendimento"]}: {int(r["unidades_em_obra"])} un.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

categorias = ['Casas/apartamentos\nentregues (2017-2026)',
              'Casas/apartamentos\nem obra (escopo social)']
valores = [total_moradias_entregues, total_obra]
cores_barras = [COR_ENTREGUE, COR_OBRA]

barras = ax.bar(categorias, valores, color=cores_barras, width=0.5, zorder=3)
for x, y in enumerate(valores):
    ax.annotate(f'{int(y)}', (x, y), textcoords='offset points',
                xytext=(0, 6), ha='center', fontsize=12, fontweight='bold')

ax.set_title('Lucas do Rio Verde: moradias entregues vs. moradias em obra', fontsize=12)
ax.set_ylabel('Casas/apartamentos')
ax.grid(True, axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

### Leitura: pipeline em obras

O conjunto de dados registra **2.000 casas em construção** dentro do escopo social estrito, em comparação com **1.084 casas/apartamentos entregues** entre 2017 e 2026. Há também 171 unidades de um empreendimento em obra marcado fora do escopo social estrito; elas não entram no indicador principal.

Essa comparação sugere um pipeline relevante, mas **obra em andamento não é moradia entregue**. Atrasos e paralisações podem alterar o calendário de conclusão, portanto o pipeline deve ser tratado como oferta potencial.

## 5. Para quem e de que tipo

A política habitacional não é homogênea: atende faixas de renda diferentes, com tipos de moradia diferentes (casa, apartamento, lote urbanizado). Esta seção decompõe a oferta total (entregue + em obra) por essas dimensões.

In [ ]:
# Resume a oferta dentro do escopo de habitação de interesse social estrito.
# O empreendimento marcado fora do escopo permanece no dataset original,
# mas não entra nestas composições.
oferta_social = oferta[oferta['interesse_social_estrito']].copy()
oferta_social['unidades_total'] = oferta_social['unidades_previstas']

por_tipo = oferta_social.groupby('tipo')['unidades_total'].sum().sort_values(ascending=False)
print('--- Unidades previstas por tipo (escopo social estrito) ---')
print(por_tipo.to_string())
print()

por_faixa = oferta_social.groupby('faixa_renda')['unidades_total'].sum().sort_values(ascending=False)
print('--- Unidades previstas por faixa de renda (escopo social estrito) ---')
print(por_faixa.to_string())
print()
print('Empreendimentos fora do escopo social estrito excluídos desta composição:',
      int((~oferta['interesse_social_estrito']).sum()))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico de pizza por tipo
paleta_tipo = ['#2c7fb8', '#7fcdbb', '#253494', '#c7e9b4']
ax1.pie(por_tipo.values, labels=por_tipo.index, autopct='%1.0f%%',
        colors=paleta_tipo[:len(por_tipo)], startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax1.set_title('Composição por tipo de moradia', fontsize=12)

# Barras por faixa
paleta_faixa = ['#253494', '#2c7fb8', '#7fcdbb', '#c7e9b4', '#edf8b1']
ax2.barh(por_faixa.index[::-1], por_faixa.values[::-1],
         color=paleta_faixa[:len(por_faixa)][::-1], zorder=3)
for i, v in enumerate(por_faixa.values[::-1]):
    ax2.annotate(f'{int(v)}', (v, i), textcoords='offset points',
                 xytext=(4, 0), va='center', fontsize=10, fontweight='bold')
ax2.set_title('Composição por faixa de renda', fontsize=12)
ax2.set_xlabel('Unidades (entregue + em obra)')
ax2.grid(True, axis='x', alpha=0.3, zorder=0)

plt.tight_layout()
plt.show()

### Leitura: predomínio de casas, foco em baixa renda

O tipo dominante é **casa** — a maior parte da oferta, puxada pelo grande empreendimento Jardim Esperança (2.000 unidades). Apartamentos aparecem em segundo (Águas do Cerrado, Parque Primavera), e os lotes urbanizados do Sonho Meu representam uma modalidade distinta: entrega de terreno pronto, não moradia construída.

Quanto à faixa de renda, o grosso da oferta se concentra nas **Faixas 1 e 2** (interesse social estrito), com apenas o Parque Primavera (171 unidades, flag `False`) atendendo renda mais alta. Isso confirma que a política habitacional do município está, de fato, voltada a quem mais precisa — coerente com o objetivo declarado dos programas.

## 6. De onde vem o dinheiro

Habitação de interesse social costuma empilhar recursos de três níveis de governo. Esta seção mostra em quantos empreendimentos cada fonte aparece — revelando o grau de articulação federativa da política.

In [ ]:
fontes = {
    'Federal\n(Minha Casa Minha Vida)': oferta_social['fonte_federal'].sum(),
    'Estadual\n(Ser Família / MT Par)': oferta_social['fonte_estadual'].sum(),
    'Municipal\n(Ser Luverdense)': oferta_social['fonte_municipal'].sum(),
}

print('Empreendimentos no escopo social estrito:', len(oferta_social))
for k, v in fontes.items():
    print(f'  {k.replace(chr(10), " ")}: {int(v)}')

tres_fontes = oferta_social[
    (oferta_social['fonte_federal']) &
    (oferta_social['fonte_estadual']) &
    (oferta_social['fonte_municipal'])
]
print(f'\nEmpreendimentos com as TRÊS fontes empilhadas: {len(tres_fontes)} de {len(oferta_social)}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

nomes = list(fontes.keys())
valores = list(fontes.values())
paleta_fontes = ['#253494', '#2c7fb8', '#7fcdbb']

barras = ax.bar(nomes, valores, color=paleta_fontes, width=0.55, zorder=3)
for x, y in enumerate(valores):
    ax.annotate(f'{int(y)}', (x, y), textcoords='offset points',
                xytext=(0, 6), ha='center', fontsize=12, fontweight='bold')

ax.set_title('Habitação social: empreendimentos por fonte de recurso\n'
             f'{len(tres_fontes)} dos {len(oferta_social)} combinam as três esferas', fontsize=12)
ax.set_ylabel('Número de empreendimentos')
ax.set_ylim(0, len(oferta_social) + 1)
ax.grid(True, axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

### Leitura: articulação federativa

No conjunto incluído no escopo social estrito, a fonte federal (Minha Casa Minha Vida) aparece em quase todos os empreendimentos. A presença simultânea de recursos federais, estaduais e municipais indica articulação entre esferas, embora a contagem de fontes não represente o valor financeiro de cada contribuição.

A base identifica quais fontes estão assinaladas em cada empreendimento; ela não permite, por si só, comparar a participação percentual de cada esfera no investimento total.

## 7. Oferta vs. fila de espera: o que os dados permitem afirmar

A base registra uma fila de **5.800 famílias em 2021** (4.000 no perfil Faixa 1). Esse dado é uma referência histórica de demanda cadastrada, não uma série anual.

O dataset também registra entregas entre 2017 e 2026. Porém, não identifica de forma individualizada quais famílias cadastradas foram atendidas. Além disso, as entregas incluem **1.084 casas/apartamentos e 287 lotes urbanizados**, que não são categorias equivalentes.

Por isso, não calculamos uma “cobertura da fila” dividindo todas as entregas pela fila de 2021, nem subtraímos unidades de famílias para estimar um déficit residual. Esses cálculos exigiriam dados de atendimento e compatibilidade temporal que não estão presentes nesta base.

In [ ]:
fila_2021 = int(demanda.iloc[0]['inscritos_total'])
fila_faixa1 = int(demanda.iloc[0]['inscritos_faixa1'])

print('=== INDICADORES DE OFERTA E DEMANDA (sem taxa de cobertura inferida) ===')
print(f'Fila cadastrada em 2021: {fila_2021:,} famílias'.replace(',', '.'))
print(f'Dessas, Faixa 1: {fila_faixa1:,} famílias'.replace(',', '.'))
print()
print('Entregas registradas entre 2017 e 2026:')
print(f'  Casas/apartamentos prontos: {total_moradias_entregues}')
print(f'  Lotes urbanizados:          {total_lotes_entregues}')
print()
print(f'Casas em construção no escopo social estrito: {total_obra}')
print(f'Unidades em obra fora desse escopo:           {total_obra_fora_escopo}')
print()
print('Limite metodológico: os dados não mostram quais famílias da fila foram atendidas.')
print('Assim, não é válido calcular cobertura ou déficit residual apenas pela divisão/subtração desses totais.')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

categorias = [
    'Casas/apartamentos\nentregues (2017-2026)',
    'Lotes urbanizados\nentregues (2017-2026)',
    'Casas em obra\n(escopo social estrito)'
]
valores = [total_moradias_entregues, total_lotes_entregues, total_obra]
cores_barras = [COR_ENTREGUE, '#7fcdbb', COR_OBRA]

barras = ax.bar(categorias, valores, color=cores_barras, width=0.58, zorder=3)
for x, y in enumerate(valores):
    ax.annotate(f'{int(y)}', (x, y), textcoords='offset points',
                xytext=(0, 6), ha='center', fontsize=12, fontweight='bold')

ax.set_title('Habitação social: entregas por categoria e pipeline em construção\n'
             'Categorias separadas; fila de 2021 não é representada como cobertura', fontsize=12)
ax.set_ylabel('Casas/apartamentos ou lotes')
ax.grid(True, axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

### Leitura: fila histórica e oferta não são a mesma medida

A fila de 5.800 famílias em 2021 contextualiza a escala da necessidade registrada naquele ano, mas não permite calcular quantas famílias continuam aguardando hoje. Sem um registro que relacione entregas a famílias atendidas e atualize a fila, o percentual de cobertura não pode ser inferido.

Separar casas/apartamentos de lotes urbanizados também evita tratar a entrega de terreno como se fosse a entrega de uma moradia pronta.

## 8. Conexão com a Camada 1: oferta vs. demanda projetada

A fila de espera mede a demanda *acumulada* (estoque). A Camada 1 projetou a demanda *anual* (fluxo): quantas moradias novas o crescimento populacional exige a cada ano.

Colocar as duas lado a lado revela se a cidade constrói no ritmo que a própria demografia impõe.

In [ ]:
# Demanda bruta anual projetada na Camada 1 para 2027-2035.
# Essa demanda abrange a necessidade total derivada do crescimento populacional,
# enquanto a oferta abaixo é somente habitação social entregue.
# A comparação é de escala e contexto, não uma estimativa direta do déficit social.
demanda_anual = {'Conservador': 1121, 'Base': 1507, 'Otimista': 1816}

oferta_media_anual = por_ano[por_ano > 0].mean()
oferta_media_periodo = por_ano.sum() / len(por_ano)

print('=== DEMANDA BRUTA TOTAL PROJETADA (2027-2035, Camada 1) ===')
for k, v in demanda_anual.items():
    print(f'  Cenário {k}: {v} novas moradias/ano')
print()
print('=== OFERTA HISTÓRICA DE HABITAÇÃO SOCIAL (casas/apartamentos) ===')
print(f'  Média nos anos com entrega:      {oferta_media_anual:.0f} unidades/ano')
print(f'  Média no período (2017-2026):    {oferta_media_periodo:.0f} unidades/ano')
print()
print('=== COMPARAÇÃO DE ESCALA (não é déficit medido) ===')
print(f'  Demanda total anual no cenário Base: {demanda_anual["Base"]} moradias/ano')
print(f'  Oferta social histórica média:       {oferta_media_periodo:.0f} moradias/ano')
print('  A diferença indica ordens de grandeza distintas, mas os escopos não são iguais.')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x_dem = [0, 1, 2]
vals_dem = [demanda_anual['Conservador'], demanda_anual['Base'], demanda_anual['Otimista']]
ax.bar(x_dem, vals_dem, color=[cores['Conservador'], cores['Base'], cores['Otimista']],
       width=0.55, zorder=3, label='Demanda total bruta projetada (Camada 1)')
for x, y in zip(x_dem, vals_dem):
    ax.annotate(f'{int(y)}', (x, y), textcoords='offset points',
                xytext=(0, 6), ha='center', fontsize=11, fontweight='bold')

ax.axhline(oferta_media_periodo, color=COR_DEMANDA, linestyle='--', linewidth=2, zorder=4,
           label=f'Oferta social histórica média: {oferta_media_periodo:.0f}/ano')

ax.set_xticks(x_dem)
ax.set_xticklabels(['Conservador', 'Base', 'Otimista'])
ax.set_title('Lucas do Rio Verde: demanda habitacional total projetada vs. oferta social histórica\n'
             'Comparação de escala, não cálculo direto de déficit')
ax.set_ylabel('Moradias por ano')
ax.set_xlabel('Cenário de demanda (Camada 1)')
ax.legend(loc='upper left')
ax.grid(True, axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

### Leitura: uma comparação de escala com escopos diferentes

A projeção da Camada 1 estima a demanda **total** por novas moradias derivada do crescimento populacional; esta camada mede a oferta histórica de **habitação de interesse social**. A demanda projetada no cenário Base fica perto de 1.507 moradias por ano entre 2027 e 2035, enquanto a oferta social registrada de casas/apartamentos entregues teve média de cerca de 108 por ano no período completo de 2017 a 2026 (ou 271 nos anos com entrega).

A diferença indica que a produção social histórica é pequena em comparação com a escala da demanda habitacional total projetada. No entanto, como os universos analisados são diferentes, essa diferença **não deve ser chamada de déficit anual de habitação social**. Para medir esse déficit seriam necessários dados sobre renda e elegibilidade, atendimento efetivo das famílias, unidades produzidas por outros segmentos e evolução atual da fila.

O pipeline social de 2.000 casas em obra é relevante, mas depende de conclusão e entrega; não deve ser contado como oferta realizada.

## 9. Limitações

1. **Dataset curado manualmente.** Construído a partir de notícias e portais oficiais, não de uma base estruturada. É auditável (toda linha tem fonte), mas pode conter empreendimentos não capturados.
2. **Fila de espera é um ponto único (2021).** Não há série histórica pública da demanda cadastrada. A evolução ano a ano exigiria pedido via LAI à MT Par/SiHab-MT.
3. **Investimento incompleto.** Só o Vida Nova II tem valor consolidado (R$ 27,9 mi, Min. Cidades). Os demais exigiriam consulta a licitações no portal de transparência ou LAI.
4. **Programa de 2013 não confirmado.** Uma iniciativa de ~2.465 casas foi anunciada em 2013 (construtora Aurora), mas as entregas efetivas não puderam ser confirmadas em fonte pública — pode haver oferta anterior a 2017 não capturada.
5. **Obra ≠ entrega.** O pipeline de 2.171 unidades é uma promessa, não um fato. Atrasos são comuns (o Vida Nova II foi paralisado e retomado).
6. **Demanda anual da Camada 1 é projeção.** Herda as incertezas daquele modelo (taxa de crescimento constante, etc.).

## 10. Síntese executiva

Entre 2017 e 2026, a base registra **1.084 casas/apartamentos entregues** e **287 lotes urbanizados** de interesse social. O total de 1.371 reúne categorias diferentes; por isso, não deve ser descrito como 1.371 casas prontas. A base registra ainda **2.000 casas em construção** dentro do escopo social estrito. Outras 171 unidades em obra aparecem em um empreendimento marcado fora desse escopo e são excluídas do indicador principal.

A fila de espera publicada para 2021 tinha 5.800 famílias, das quais 4.000 no perfil Faixa 1. Como os dados não identificam quais famílias foram atendidas, não calculamos percentual de cobertura nem déficit residual a partir da subtração entre famílias e unidades entregues.

A Camada 1 projeta demanda habitacional total bruta de cerca de **1.121 a 1.816 moradias por ano** entre 2027 e 2035, conforme o cenário de crescimento. A oferta social histórica de casas/apartamentos teve média aproximada de 108 por ano no período completo (2017–2026). A comparação evidencia uma diferença de escala, mas não é uma estimativa direta do déficit de habitação social porque demanda total e oferta social têm escopos distintos.

A principal conclusão é metodológica e substantiva: a base pública estruturada permite medir entregas, composição e pipeline, mas uma avaliação rigorosa de atendimento social requer dados atualizados da fila, dos beneficiários atendidos e de unidades disponíveis fora dos empreendimentos catalogados.